# CardioScore — Full Public Validation Campaign v2 (resumable)

Use **CPU** and **Run all**. This is the corrected campaign runner after auditing the first returned bundle.

It uses the existing persistent Drive root `MyDrive/CardioScore_validation_campaign_v1/`, so already downloaded raw files are reused. Runner-schema v2 intentionally re-evaluates the previous v1 stage checkpoints once because the first run exposed acquisition/dispatch bugs.

Fixes in v2:
- makes the CardioScore checkout importable in the already-running Colab kernel (fixes PyMEA `ModuleNotFoundError`);
- downloads Dryad datasets through the supported dataset → version → files API, with a legacy landing-page fallback;
- accepts the actual Figshare Geiger workbook filename instead of requiring one hard-coded alias;
- prevents Ballouz from reporting a completed analysis when zero MATLAB files were acquired;
- clears stale blocked markers when a rerun succeeds;
- preserves Drive resume/caching and produces the same final results ZIP.


In [ ]:
#@title 1) Campaign options
import os
AUTO_DOWNLOAD_RESULTS = True  #@param {type:"boolean"}
PYMEA_HELDOUT_WINDOWS = 40  #@param {type:"integer"}
PYMEA_HELDOUT_SEED = 20261019  #@param {type:"integer"}
FORCE_STAGES = ""  # optional, e.g. geiger_2025_nam:analysis
os.environ["CARDIOSCORE_CAMPAIGN_AUTO_DOWNLOAD"] = "1" if AUTO_DOWNLOAD_RESULTS else "0"
os.environ["CARDIOSCORE_CAMPAIGN_PYMEA_WINDOWS"] = str(PYMEA_HELDOUT_WINDOWS)
os.environ["CARDIOSCORE_CAMPAIGN_PYMEA_SEED"] = str(PYMEA_HELDOUT_SEED)
os.environ["CARDIOSCORE_CAMPAIGN_FORCE"] = FORCE_STAGES
print("Configured. Existing Drive cache will be reused.")


In [ ]:
#@title 2) Run the SHA-pinned corrected campaign
import hashlib, urllib.request
RUNNER_SHA = "b1def2d24302827f6b391e1c5054606dbd9f04e1"
RUNNER_URL = "https://raw.githubusercontent.com/Virelion-Biotech/Virelion-CardioScore/b1def2d24302827f6b391e1c5054606dbd9f04e1/scripts/validation/run_validation_campaign_colab.py"
source = urllib.request.urlopen(RUNNER_URL).read()
print("Runner commit:", RUNNER_SHA)
print("Runner SHA-256:", hashlib.sha256(source).hexdigest())
code = compile(source, "run_validation_campaign_colab.py", "exec")
ns = {"__name__": "__main__", "__file__": "run_validation_campaign_colab.py"}
try:
    exec(code, ns)
except SystemExit as exc:
    if exc.code not in (0, None):
        raise


## Send back

After the run, send `CardioScore_public_validation_campaign_results.zip` back to ChatGPT.

If PyMEA succeeds, the result bundle should also contain a blinded annotation package. That human annotation step remains necessary for an independent beat-detection validation claim.
